# Lab 04: Linear Scores and Multiclass SVM Loss



By the end of this lab you will compute linear class scores, implement the
loop-based multiclass SVM loss and gradient, and verify the gradient with
finite differences.


## 1. Setup

Before running any cell, select the **Python (DLCV 2026)** kernel. If it is
missing, follow `Labs/README.md` from the repository root.

All exercises run on CPU and require only PyTorch. This notebook uses
IPython's autoreload extension, as in the original assignment. Save
`linear_classifier.py`, then rerun the relevant validation cell; you do not
need to restart the kernel. The setup cell prints the implementation path so
you can confirm that you are editing the correct file.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import torch

import dlcv2026
import linear_classifier

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

print("PyTorch:", torch.__version__)
print("Implementation:", Path(linear_classifier.__file__).resolve())

## 2. Linear scores

For a minibatch, a linear classifier computes $S=XW+b$. Each row of $S$
contains one score per class. Run the fixture and identify the predicted class
for each example before looking at the printed result.


In [ ]:
X_demo = torch.tensor([[1.0, 2.0], [-1.0, 1.0]])
W_demo = torch.tensor([[0.5, -1.0, 0.0], [1.0, 0.5, -0.5]])
b_demo = torch.tensor([0.1, 0.0, 0.2])

scores_demo = linear_classifier.linear_scores(X_demo, W_demo, b_demo)
predictions_demo = scores_demo.argmax(dim=1)
print("scores:\n", scores_demo)
print("predicted classes:", predictions_demo.tolist())

assert scores_demo.shape == (2, 3)
assert predictions_demo.dtype == torch.int64

## 3. Multiclass SVM loss and gradient

For example $i$, correct class $y_i$, and margin $\Delta=1$:

$$
L_i = \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + \Delta).
$$

Open `linear_classifier.py` and complete `svm_loss_naive`. Use explicit
loops. When a margin is positive, update both the loss and the two affected
columns of the gradient. Complete all four TODOs: accumulate positive
margins, update the two gradient columns, average over the minibatch, and
return `loss, dW`.

For Lab 04, use the unregularized objective:

$$
L = \frac{1}{N}\sum_i L_i.
$$

Keep `reg=0.0` throughout this lab. Lecture 3B and Lab 05 introduce L2
regularization.


In [ ]:
# Save linear_classifier.py, then rerun this cell.
X = torch.tensor([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])
W = torch.tensor([[1.0, -1.0, 0.0], [0.0, 1.0, -1.0]])
y = torch.tensor([2, 0, 1], dtype=torch.int64)

loss, gradient = linear_classifier.svm_loss_naive(
    W.clone(), X, y, reg=0.0
)
print("unregularized loss:", float(loss))
print("gradient:\n", gradient)

assert loss.ndim == 0
assert gradient.shape == W.shape
assert torch.allclose(loss, torch.tensor(2.0), atol=1e-12)

## 4. Numerical gradient check

Finite differences provide an independent check of your analytic gradient.
Most relative errors should be below $10^{-5}$. A few larger values can occur
exactly at the nondifferentiable hinge boundary; the random fixture avoids that
case in normal runs.


In [ ]:
help(dlcv2026.grad_check_sparse)

torch.manual_seed(4)
X_check = torch.randn(7, 5)
W_check = 0.1 * torch.randn(5, 4)
y_check = torch.tensor([0, 1, 2, 3, 0, 1, 2], dtype=torch.int64)

loss_check, grad_check = linear_classifier.svm_loss_naive(
    W_check, X_check, y_check, reg=0.0
)
f = lambda weights: linear_classifier.svm_loss_naive(
    weights, X_check, y_check, reg=0.0
)[0]
errors = dlcv2026.grad_check_sparse(
    f, W_check, grad_check, num_checks=10, h=1e-6
)
print("maximum relative error:", max(errors))
assert max(errors) < 1e-5

## 5. Wrap-up

Answer briefly in a new Markdown cell:

1. What changes in the loss when the correct-class score increases by 0.5?
2. Why does one positive margin update two columns of $W$?
3. How would increasing the margin $\Delta$ change which examples
   contribute to the loss?

**Optional homework:** implement a squared hinge-loss variant and explain how
its gradient differs from the standard hinge loss.


## Wrap-up answers

**1. What changes in the loss when the correct-class score increases by 0.5?**
Every margin of example $i$ has the form $s_j - s_{y_i} + \Delta$, so raising $s_{y_i}$ by 0.5 lowers each of those margins by 0.5. Each class $j$ whose margin was positive therefore contributes up to 0.5 less (a margin that was already below 0.5 just drops to 0 and stops contributing), and classes with margin $\le 0$ are unaffected. The loss of that example decreases by at most $0.5 \times$ (number of violating classes), and it never goes below 0. If no margin was positive, the loss does not change. In the mean loss this decrease is divided by $N$.

**2. Why does one positive margin update two columns of $W$?**
A positive margin $m = s_j - s_{y_i} + \Delta$ involves two scores, $s_j = x_i \cdot W[:, j]$ and $s_{y_i} = x_i \cdot W[:, y_i]$. Therefore $\partial m / \partial W[:, j] = +x_i$ and $\partial m / \partial W[:, y_i] = -x_i$. The gradient pushes column $j$ (the wrong class) to give this example a lower score, and column $y_i$ (the correct class) to give it a higher score. All other columns do not appear in this margin, so their gradient is zero.

**3. How would increasing the margin $\Delta$ change which examples contribute to the loss?**
A larger $\Delta$ demands a larger gap between the correct-class score and every other score. Terms that were satisfied with a small gap ($0 < s_{y_i} - s_j < \Delta$ for the new $\Delta$) now have a positive margin, so more example/class pairs contribute to the loss and to the gradient, and the loss values are larger. A smaller $\Delta$ does the opposite: fewer pairs are violated and fewer examples contribute. (With a regularized model $\Delta$ and the weight scale are tied together, which is discussed in Lecture 3B / Lab 05.)
